# VQE for molecular binding energies on Amazon Braket: from SV1 to IonQ Forte

This notebook demonstrates a complete, reproducible Variational Quantum Eigensolver (VQE) workflow for molecular electronic structure on Amazon Braket, taken all the way to a real quantum processor. It is aimed at researchers and computational chemists who are comfortable with quantum circuits and basic quantum chemistry (active spaces, basis sets, Hartree-Fock) but have not run VQE on Amazon Braket before.

**What you will do here**
1. Build a molecular electronic Hamiltonian from atomic coordinates with an active space (PennyLane qchem, Jordan-Wigner mapping).
2. Optimize a particle-conserving `AllSinglesDoubles` ansatz on the free local statevector simulator.
3. Evaluate the converged energy on Amazon Braket SV1 (optional, billable).
4. Combine fragment energies into a binding (reaction) energy.
5. Inspect a real IonQ Forte hardware run for H2 (raw and error-mitigated) as an honest reality check, and optionally submit your own.

**What is new here vs already-published content.** The VQE method, the Jordan-Wigner mapping, the UCCSD/AllSinglesDoubles ansatz, and the individual error-mitigation techniques are all covered elsewhere (see the links at the end). What this entry adds is (a) a single, reproducible pipeline from a molecular geometry to a binding energy on the open PennyLane + Braket stack, and (b) a real IonQ Forte hardware result with honest error analysis, rather than a simulator-only demonstration.

**Scope and limits.** The chemistry here is illustrative, not converged to experimental accuracy: minimal STO-3G basis and small active spaces (CAS(2,2) for H2, CAS(4,4) for H2O). The rigorous claim is that VQE reproduces the exact (FCI) energy *within the chosen active space*, which is what we verify throughout. Mapping an active-space energy difference to a real binding free energy needs basis-set extrapolation, a larger active space, geometry/solvation, and entropic terms, which are out of scope for this example.

## Prerequisites

Install the algorithm library and the extra open-source dependencies for this notebook:

```bash
pip install amazon-braket-algorithm-library
pip install -r requirements.txt
```

The local-simulator cells run with no AWS account and no cost. Only the SV1 cell and the live QPU cell touch Amazon Braket, are billable, and need AWS credentials configured for `us-east-1`. Both are disabled by default via the flags below, so `Cells > Run All` completes end to end on the free local simulator.

In [ ]:
import json
from pathlib import Path

import pennylane as qml

from braket.experimental.algorithms.vqe_chemistry import (
    build_molecular_hamiltonian,
    run_vqe_chemistry,
    get_vqe_chemistry_results,
    compute_binding_energy,
    exact_ground_state_energy,
    SV1_ARN,
)

# Billable cells are off by default so the notebook runs free, end to end, on the local simulator.
RUN_ON_SV1 = False        # set True to evaluate the converged energy on Amazon Braket SV1 (billable)
SUBMIT_TO_QPU = False     # set True to submit a live H2 energy to IonQ Forte (billable, slow)

IONQ_FORTE_ARN = "arn:aws:braket:us-east-1::device/qpu/ionq/Forte-Enterprise-1"

## 1. Build a molecular Hamiltonian

We start with water in a CAS(4,4) active space: 4 active electrons in 4 active spatial orbitals, which maps to 8 qubits under Jordan-Wigner. `build_molecular_hamiltonian` returns the qubit Hamiltonian along with the Hartree-Fock reference state and the single/double excitation lists that wire up the ansatz.

In [ ]:
# Water at its equilibrium geometry (atomic units, Bohr).
water_symbols = ["O", "H", "H"]
water_geometry = [
    [0.0000, 0.0000, 0.2217],
    [0.0000, 1.4309, -0.8867],
    [0.0000, -1.4309, -0.8867],
]

ham = build_molecular_hamiltonian(water_symbols, water_geometry, active_electrons=4, active_orbitals=4)
print(f"Qubits      : {ham['n_qubits']}")
print(f"Pauli terms : {ham['n_pauli_terms']}")
print(f"Parameters  : {ham['n_parameters']} "
      f"({len(ham['singles'])} singles, {len(ham['doubles'])} doubles)")
print(f"HF state    : {[int(x) for x in ham['hf_state']]}")

## 2. Run VQE on the local simulator

`run_vqe_chemistry` optimizes the ansatz parameters on the local statevector simulator (deterministic, exact adjoint gradients) and compares the result to the exact active-space energy (FCI) obtained by direct diagonalization. For CAS(4,4) water, VQE should land far inside chemical accuracy (1.6 mHa).

In [ ]:
water_result = run_vqe_chemistry(
    water_symbols, water_geometry, active_electrons=4, active_orbitals=4, max_iterations=80
)
get_vqe_chemistry_results(water_result)

## 3. Evaluate on Amazon Braket SV1 (optional, billable)

SV1 is the on-demand managed statevector simulator. In analytic mode (`shots=0`) it returns exact expectation values, so the SV1 energy should match the local result to machine precision. This cell is gated by `RUN_ON_SV1` because it submits a billable task and needs AWS credentials.

In [ ]:
if RUN_ON_SV1:
    water_sv1 = run_vqe_chemistry(
        water_symbols, water_geometry, active_electrons=4, active_orbitals=4,
        max_iterations=80, device_arn=SV1_ARN, shots=0,
    )
    print(get_vqe_chemistry_results(water_sv1))
else:
    print("RUN_ON_SV1 is False. Set it to True (with AWS credentials) to evaluate on SV1.")
    print("Reference: a verified SV1 analytic run gave -74.97045946 Ha, 0.0005 mHa from FCI.")

## 4. Binding energy from fragments

A binding (or reaction) energy is a difference of total energies: `dE = E(complex) - sum(E(fragments))`. Because each piece is computed independently, errors that are common to the fragments cancel.

As a runnable demonstration, we compute a weakly interacting (H2)2 dimer live on the local simulator: the complex is two H2 molecules placed about 6 Bohr apart (CAS(4,4), 8 qubits), and each H2 fragment uses a size-consistent CAS(2,2) active space (4 qubits). The result is a small, weakly repulsive value near +0.2 kcal/mol. At the minimal STO-3G basis dispersion is not captured, so this illustrates the fragment bookkeeping rather than a converged interaction energy. This is the runnable binding demonstration in this notebook (everything is computed live).

For a drug-relevant example, the cell after it reports a covalent thia-Michael addition (a cysteine-warhead model: H2S + acrylonitrile to the covalent adduct). Those numbers are a precomputed reference, not computed in this notebook: the sulfur-containing species need an external basis set (basis-set-exchange) and a PySCF Hartree-Fock step, so they are regenerated with the companion script `sv1_covalent.py` rather than inside Run All. The values were produced by the same VQE pipeline and evaluated on Amazon Braket SV1.

In [ ]:
# Real (H2)2 dimer binding energy, computed live (all local hydrogen, no extra dependencies).
# Two H2 molecules are placed collinearly along z about 6 Bohr apart, so they weakly interact.
# complex = 4 H atoms at CAS(4,4) -> 8 qubits; each H2 fragment at CAS(2,2) -> 4 qubits.
h2_geom = [[0.0, 0.0, 0.0], [0.0, 0.0, 1.4]]
dimer_geom = [[0.0, 0.0, 0.0], [0.0, 0.0, 1.4], [0.0, 0.0, 6.0], [0.0, 0.0, 7.4]]

complex_res = run_vqe_chemistry(["H", "H", "H", "H"], dimer_geom,
                                active_electrons=4, active_orbitals=4,
                                max_iterations=80, verbose=False)
monomer_res = run_vqe_chemistry(["H", "H"], h2_geom,
                                active_electrons=2, active_orbitals=2,
                                max_iterations=60, verbose=False)

# Both H2 fragments have the same internal geometry, so they share one energy.
binding = compute_binding_energy(complex_res, [monomer_res, monomer_res])
print(f"E(complex, CAS(4,4))     : {complex_res.vqe_energy:.8f} Ha")
print(f"E(H2 fragment, CAS(2,2)) : {monomer_res.vqe_energy:.8f} Ha")
print(f"(H2)2 binding energy     : {binding['binding_energy_kcal_per_mol']:+.2f} kcal/mol "
      f"(weakly repulsive; STO-3G captures no dispersion)")

In [ ]:
# Covalent thia-Michael result, loaded from a cached data file (not computed in this notebook).
# Regenerate the file with the companion scripts: thia_michael.py (local FCI/VQE) and
# sv1_covalent.py (Amazon Braket SV1). The sulfur species need RDKit + PySCF + basis-set-exchange.
# The notebook reads the file rather than hardcoding the numbers; it only does the bookkeeping arithmetic.
covalent = json.loads(Path("covalent_cached.json").read_text())
sp = covalent["species"]
dE_fci = (sp["adduct"]["fci_ha"] - sp["H2S"]["fci_ha"] - sp["acrylonitrile"]["fci_ha"]) \
    * covalent["hartree_to_kcal_per_mol"]
max_dev = max(s["vqe_fci_dev_mha"] for s in sp.values())
print("[cached reference from covalent_cached.json; regenerate with thia_michael.py / sv1_covalent.py]")
print(f"Covalent binding energy (FCI): {dE_fci:.2f} kcal/mol  (VQE agrees to ~0.01 kcal/mol)")
print(f"Max VQE-vs-FCI per-species deviation: {max_dev:.4f} mHa (better than 0.02 mHa)")

## 5. A real IonQ Forte hardware run (reality check)

Simulators are noiseless. To show what current hardware actually returns, we ran the smallest meaningful case, H2 on 4 qubits with a single `DoubleExcitation` gate, on **IonQ Forte Enterprise** via Amazon Braket. We chose IonQ because its all-to-all connectivity runs the non-local excitation gate without SWAP routing overhead.

The results below are loaded from `ionq_forte_h2_cached.json` so this notebook runs without resubmitting billable QPU tasks. Each energy was independently recomputed from the completed Amazon Braket S3 task data (`verify_result.py`, `verify_mitigated.py`).

In [ ]:
cache = json.loads(Path("ionq_forte_h2_cached.json").read_text())
fci = cache["fci_energy_ha"]
print(f"Device : {cache['device_arn']}")
print(f"System : {cache['molecule']} {cache['basis']} {cache['active_space']}, {cache['n_qubits']} qubits\n")
print(f"{'method':<42}{'energy (Ha)':>14}{'error vs FCI (mHa)':>22}")
print(f"{'exact (FCI)':<42}{fci:>14.8f}{0.0:>22.2f}")
print(f"{'noiseless VQE (local / SV1)':<42}{fci:>14.8f}{0.0:>22.2f}")
for run in cache["runs"]:
    print(f"{'IonQ Forte ' + run['label']:<42}{run['energy_ha']:>14.8f}{run['error_vs_fci_mha']:>22.2f}")
print(f"\n{cache['notes']}")

**Reading the result.** The variational principle keeps the hardware energy a valid upper bound on FCI, so the error is positive. The raw run sits about 50 mHa above FCI, and IonQ debiasing recovers only about 1 mHa. That small recovery is the informative part: it tells us the residual error is dominated by incoherent two-qubit-gate noise and decoherence (which debiasing does not remove) rather than coherent systematic bias. Reaching chemical accuracy on this circuit needs the fuller mitigation stack: readout-error mitigation, symmetry post-selection (keep only the 2-electron outcomes), and zero-noise extrapolation via Braket program sets with Mitiq.

### 5b. Submit your own H2 energy to IonQ Forte (optional, billable, slow)

Set `SUBMIT_TO_QPU = True` to run the live evaluation. The QPU has queue and availability windows, so this can take minutes to hours and is billable. It evaluates the energy once at the locally-optimized angle using a minimal single-`DoubleExcitation` ansatz (the shallowest chemically-correct H2 circuit).

In [ ]:
if SUBMIT_TO_QPU:
    from scipy.optimize import minimize
    h2_data = build_molecular_hamiltonian(["H", "H"], h2_geom, 2, 2)
    H, n_q, hf = h2_data["hamiltonian"], h2_data["n_qubits"], h2_data["hf_state"]

    def minimal_ansatz(theta):
        qml.BasisState(hf, wires=range(n_q))
        qml.DoubleExcitation(theta, wires=range(n_q))

    # Optimize the single angle locally (free), then evaluate once on hardware.
    local = qml.device("default.qubit", wires=n_q)
    @qml.qnode(local, diff_method="adjoint")
    def local_energy(theta):
        minimal_ansatz(theta)
        return qml.expval(H)
    theta_opt = float(minimize(lambda x: float(local_energy(x[0])), x0=[0.0],
                               method="COBYLA", options={"maxiter": 200}).x[0])

    qpu = qml.device("braket.aws.qubit", device_arn=IONQ_FORTE_ARN, wires=n_q, shots=2000)
    @qml.qnode(qpu, diff_method=None)
    def qpu_energy(theta):
        minimal_ansatz(theta)
        return qml.expval(H)

    e_fci = exact_ground_state_energy(H, n_q)
    e_hw = float(qpu_energy(theta_opt))
    print(f"IonQ Forte H2 energy: {e_hw:.8f} Ha  (FCI {e_fci:.8f} Ha)")
    print(f"Error vs FCI: {(e_hw - e_fci) * 1000:.2f} mHa")
else:
    print("SUBMIT_TO_QPU is False. Set it to True (with AWS credentials) to run on IonQ Forte.")

## 6. Scaling and where this goes next

Qubit count scales as twice the number of active spatial orbitals: CAS(4,4) water is 8 qubits; a CAS(12,18) active site is 36 qubits; CAS(20,40) is 80 qubits. IonQ Forte (36 qubits) and Rigetti Cepheus (108 qubits) can hold these circuits, but the H2 reality check shows that gate noise, not qubit count, is the binding constraint today. The practical path to drug-relevant systems is: shallower adaptive ansatze (ADAPT-VQE), the full error-mitigation stack, and quantum/classical embedding so only the strongly correlated core sits on the QPU.

## Companion scripts (reproduce every number)
These ship alongside this notebook in the same folder. None of them contain account IDs, task ARNs, or credentials.
- `vqe_chemistry.py`: the algorithm module used above (installed via the package).
- `sv1_covalent.py`: the covalent thia-Michael species on Amazon Braket SV1.
- `h2_ionq_forte.py`, `h2_ionq_mitigated.py`: the raw and error-mitigated IonQ Forte runs (gated behind `--submit`).
- `verify_result.py`, `verify_mitigated.py`: independent recomputation of the hardware energies from completed tasks. They take your own task ARNs as CLI arguments or via the `BRAKET_TASK_ARNS` environment variable, so no account-specific data is committed.

## References and further reading
_TODO (author accuracy pass): confirm each citation and link before submitting the PR._
- PennyLane VQE in quantum chemistry tutorial (variational principle, ansatz construction).
- Amazon Braket VQE / quantum chemistry example notebooks.
- Amazon Braket error mitigation: program sets + Mitiq; adaptive shot allocation (this repo).
- IonQ debiasing and sharpening documentation.
- Peruzzo et al., "A variational eigenvalue solver on a photonic quantum processor" (2014).
- Grimsley et al., ADAPT-VQE (2019).
- _TODO: verify and cite the Kvantify Qrunch and IonQ/AstraZeneca references if retained._